# Figuras de Decomposição Wavelet — Haar, Db4, Morlet, Coiflet

Gera quatro figuras para o Cap. 3 da tese:
- `fig_decomp_haar.pdf`
- `fig_decomp_db4.pdf`
- `fig_decomp_morlet.pdf`
- `fig_decomp_coiflet.pdf`

Cada figura tem dois painéis: funções ψ(t)/φ(t) (coluna esquerda) e exemplo de decomposição do sinal PETR4 (coluna direita).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pywt
import csv
from datetime import datetime
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams.update({'font.family': 'serif', 'font.size': 9})

OUTPUT_DIR = Path('/Users/fteodoro/Dropbox/Doutorado/Tese/figuras')
DATA_PATH  = Path('/Users/fteodoro/Dropbox/Doutorado/Fontes/LearnableWaveletLayer/data/PETR4.SA.csv')

## Leitura dos dados PETR4 (jan/2020 – dez/2021, 256 pregões)

In [ ]:
dates_all, close_all = [], []
with open(DATA_PATH, newline='') as f:
    for row in csv.DictReader(f):
        try:
            dates_all.append(datetime.strptime(row['Date'], '%Y-%m-%d'))
            close_all.append(float(row['Close']))
        except (ValueError, KeyError):
            pass

# 2020-2021 → ~490 pregões; usa 256 (potência de 2, ideal para DWT)
start, end = datetime(2020, 1, 1), datetime(2021, 12, 31)
mask = [start <= d <= end for d in dates_all]
dates_full = [d for d, m in zip(dates_all, mask) if m]
close_full = np.array([c for c, m in zip(close_all, mask) if m])
log_ret_full = np.diff(np.log(close_full))
dates_full   = dates_full[1:]

N = 256
signal  = log_ret_full[:N]
dates_s = dates_full[:N]

print(f'Sinal: {N} amostras, {dates_s[0].date()} → {dates_s[-1].date()}')

## Funções auxiliares

In [ ]:
def month_ticks(dates_list, n=None):
    """Posições de tick no início de cada mês."""
    if n is None:
        n = len(dates_list)
    tks, lbs, prev = [], [], None
    for i, d in enumerate(dates_list[:n]):
        if (d.year, d.month) != prev:
            tks.append(i)
            lbs.append(d.strftime("%b'%y"))
            prev = (d.year, d.month)
    # Mostra só a cada 2 meses para não encher
    tks2, lbs2 = tks[::2], lbs[::2]
    return tks, lbs, tks2, lbs2

## Figura DWT — Haar, Db4, Coiflet

Layout: coluna esquerda = φ(t) + ψ(t) | coluna direita = 5 painéis empilhados (sinal + cA₃ + cD₃ + cD₂ + cD₁)

In [ ]:
def make_dwt_figure(wavelet_name, display_name, out_file, level=3):
    wav = pywt.Wavelet(wavelet_name)
    res = wav.wavefun(level=10)
    if len(res) == 3:
        phi, psi, x_w = res
    else:
        phi, psi_d, phi_r, psi_r, x_w = res
        phi, psi = phi_r, psi_d   # usa reconstituição

    coeffs = pywt.wavedec(signal, wavelet_name, level=level)
    # coeffs = [cA_L, cD_L, cD_{L-1}, ..., cD_1]

    tks, lbs, tks2, lbs2 = month_ticks(dates_s, N)

    n_rows = level + 2   # Original + cA_L + L×cD
    fig = plt.figure(figsize=(13, 8))
    fig.suptitle(display_name, fontsize=12, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(1, 2, figure=fig,
                           width_ratios=[1, 3.5], wspace=0.28)

    # ── Coluna esquerda: funções ψ e φ ──────────────────────────────────
    gs_left = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0],
                                               hspace=0.5)
    ax_phi = fig.add_subplot(gs_left[0])
    ax_psi = fig.add_subplot(gs_left[1])

    ax_phi.plot(x_w, phi, color='#1565C0', lw=1.6)
    ax_phi.axhline(0, color='gray', lw=0.5, ls='--')
    ax_phi.set_title(r'Função de escala  $\phi(t)$', fontsize=8.5)
    ax_phi.set_xlabel('$t$', fontsize=8); ax_phi.grid(alpha=0.3, ls=':')
    ax_phi.tick_params(labelsize=7)

    ax_psi.plot(x_w, psi, color='#C62828', lw=1.6)
    ax_psi.axhline(0, color='gray', lw=0.5, ls='--')
    ax_psi.set_title(r'Wavelet mãe  $\psi(t)$', fontsize=8.5)
    ax_psi.set_xlabel('$t$', fontsize=8); ax_psi.grid(alpha=0.3, ls=':')
    ax_psi.tick_params(labelsize=7)

    # ── Coluna direita: decomposição DWT ─────────────────────────────────
    gs_right = gridspec.GridSpecFromSubplotSpec(n_rows, 1,
                                                subplot_spec=gs[1],
                                                hspace=0.08)

    labels = (['Sinal original'] +
              [f'$cA_{{{level}}}$ (aprox. nível {level})'] +
              [f'$cD_{{{level - i}}}$ (detalhe nível {level - i})'
               for i in range(level)])
    colors = ['#1976D2', '#6A1B9A', '#E65100', '#2E7D32', '#AD1457']
    data_list = [signal] + list(coeffs)

    axes_r = []
    for i in range(n_rows):
        ax = fig.add_subplot(gs_right[i])
        axes_r.append(ax)
        d = data_list[i]
        x_c = np.linspace(0, N - 1, len(d))
        ax.plot(x_c, d, color=colors[i % len(colors)], lw=0.75)
        ax.axhline(0, color='gray', lw=0.4, ls='--', alpha=0.6)
        ax.set_xlim(0, N - 1)
        ax.set_ylabel(labels[i], fontsize=7.5, labelpad=4)
        ax.set_xticks(tks)
        ax.grid(alpha=0.25, ls=':')
        ax.tick_params(labelsize=7)
        if i < n_rows - 1:
            ax.set_xticklabels([])
        else:
            ax.set_xticklabels([lbs[j] if j in [tks.index(t) for t in tks2]
                                else '' for j, t in enumerate(tks)],
                               fontsize=7.5, rotation=30, ha='right')
            ax.set_xlabel('PETR4 — log-retornos diários (jan/2020 – dez/2021)', fontsize=8.5)

    axes_r[0].set_title(f'Decomposição DWT — {wavelet_name.upper()}, {level} níveis',
                        fontsize=9)

    fig.savefig(OUTPUT_DIR / out_file, dpi=150, bbox_inches='tight')
    print(f'Salvo: {out_file}')
    plt.show()
    plt.close()

## Figura Morlet — CWT

Layout: coluna esquerda = ψ(t) real/imag + envoltória Gaussiana | coluna direita = sinal (128 dias) + escalograma CWT

In [ ]:
def make_morlet_figure(out_file):
    # Wavelet Morlet analítica
    t_w  = np.linspace(-4, 4, 800)
    w0   = 6.0
    psi  = np.pi**(-0.25) * np.exp(1j * w0 * t_w) * np.exp(-t_w**2 / 2)
    env  = np.exp(-t_w**2 / 2)

    # CWT num trecho de 128 dias (jan–jul 2020)
    n_cwt = 128
    sig_cwt   = signal[:n_cwt]
    dates_cwt = dates_s[:n_cwt]

    scales = np.arange(1, 64)
    coeffs_cwt, _ = pywt.cwt(sig_cwt, scales, 'morl')
    power = np.abs(coeffs_cwt)**2

    tks, lbs, tks2, lbs2 = month_ticks(dates_cwt, n_cwt)

    fig = plt.figure(figsize=(13, 7))
    fig.suptitle('Wavelet de Morlet', fontsize=12, fontweight='bold', y=1.01)

    gs = gridspec.GridSpec(1, 2, figure=fig,
                           width_ratios=[1, 3.5], wspace=0.30)

    # ── Coluna esquerda ──────────────────────────────────────────────────
    gs_left = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[0],
                                               hspace=0.5)
    ax_psi = fig.add_subplot(gs_left[0])
    ax_env = fig.add_subplot(gs_left[1])

    ax_psi.plot(t_w, psi.real, color='#1565C0', lw=1.5, label='$\\mathrm{Re}\\{\\psi\\}$')
    ax_psi.plot(t_w, psi.imag, color='#C62828', lw=1.5, ls='--',
                label='$\\mathrm{Im}\\{\\psi\\}$')
    ax_psi.fill_between(t_w,  env, 0, alpha=0.10, color='gray')
    ax_psi.fill_between(t_w, -env, 0, alpha=0.10, color='gray')
    ax_psi.axhline(0, color='gray', lw=0.5, ls='--')
    ax_psi.set_title(r'Wavelet mãe  $\psi(t)$', fontsize=8.5)
    ax_psi.set_xlabel('$t$', fontsize=8)
    ax_psi.legend(fontsize=7, loc='upper right', ncol=2)
    ax_psi.grid(alpha=0.3, ls=':'); ax_psi.tick_params(labelsize=7)

    ax_env.plot(t_w, env, color='#2E7D32', lw=1.5)
    ax_env.set_title(r'Envoltória Gaussiana $e^{-t^2/2}$', fontsize=8.5)
    ax_env.set_xlabel('$t$', fontsize=8)
    ax_env.grid(alpha=0.3, ls=':'); ax_env.tick_params(labelsize=7)
    ax_env.set_ylim(0, 1.15)

    # ── Coluna direita: sinal + escalograma ──────────────────────────────
    gs_right = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=gs[1],
                                               height_ratios=[1, 2.5],
                                               hspace=0.08)
    ax_sig  = fig.add_subplot(gs_right[0])
    ax_scal = fig.add_subplot(gs_right[1])

    ax_sig.plot(np.arange(n_cwt), sig_cwt, color='#1976D2', lw=0.85)
    ax_sig.axhline(0, color='gray', lw=0.5, ls='--')
    ax_sig.set_xlim(0, n_cwt - 1)
    ax_sig.set_ylabel('Log-retorno', fontsize=8)
    ax_sig.set_title('Sinal + Escalograma CWT — PETR4 (jan–jul 2020)', fontsize=9)
    ax_sig.set_xticks(tks); ax_sig.set_xticklabels([])
    ax_sig.grid(alpha=0.25, ls=':'); ax_sig.tick_params(labelsize=7)

    x_edges = np.arange(-0.5, n_cwt + 0.5)
    y_edges = np.arange(0.5, len(scales) + 1.5)
    pcm = ax_scal.pcolormesh(x_edges, y_edges, power, cmap='plasma', shading='flat')
    ax_scal.set_yscale('log')
    ax_scal.set_ylim(1, len(scales))
    ax_scal.set_ylabel('Escala (dias)', fontsize=8)
    ax_scal.set_xlim(0, n_cwt - 1)
    ax_scal.set_xticks(tks)
    ax_scal.set_xticklabels([lbs[j] if tks[j] in tks2 else ''
                              for j in range(len(tks))],
                             fontsize=7.5, rotation=30, ha='right')
    ax_scal.set_xlabel('PETR4 — log-retornos diários (jan–jul 2020)', fontsize=8.5)
    ax_scal.tick_params(labelsize=7)

    # Colorbar alinhado com ax_scal; eixo fantasma em ax_sig para igualar larguras
    div_sig  = make_axes_locatable(ax_sig)
    cax_sig  = div_sig.append_axes("right", size="3%", pad=0.06)
    cax_sig.set_visible(False)

    div_scal = make_axes_locatable(ax_scal)
    cax_scal = div_scal.append_axes("right", size="3%", pad=0.06)
    fig.colorbar(pcm, cax=cax_scal).set_label('$|W_x(a,b)|^2$', fontsize=8)

    fig.savefig(OUTPUT_DIR / out_file, dpi=150, bbox_inches='tight')
    print(f'Salvo: {out_file}')
    plt.show()
    plt.close()

## Geração de todas as figuras

In [ ]:
make_dwt_figure('haar',  'Wavelet de Haar',               'fig_decomp_haar.pdf',    level=3)
make_dwt_figure('db4',   'Wavelet de Daubechies (db4)',   'fig_decomp_db4.pdf',     level=3)
make_morlet_figure(                                         'fig_decomp_morlet.pdf')
make_dwt_figure('coif2', 'Wavelet Coiflet (coif2)',        'fig_decomp_coiflet.pdf', level=3)

print('\nTodas as figuras geradas com sucesso.')